In [1]:
import sys
import time

sys.path.append("..")

import numpy as np
import pandas as pd

from common import *

bench = {
    lang: load_benchmark(lang)
    for lang in LANGS
}

print("Languages:", LANGS)

[as] Processed benchmark not found.
[as] Loading from Hugging Face...


[as] Benchmark created:
  Queries: 1000
  Corpus:  1000
[bn] Processed benchmark not found.
[bn] Loading from Hugging Face...
[bn] Benchmark created:
  Queries: 1000
  Corpus:  1000
[gu] Processed benchmark not found.
[gu] Loading from Hugging Face...
[gu] Benchmark created:
  Queries: 999
  Corpus:  999
[hi] Processed benchmark not found.
[hi] Loading from Hugging Face...
[hi] Benchmark created:
  Queries: 1000
  Corpus:  1000
[kn] Processed benchmark not found.
[kn] Loading from Hugging Face...
[kn] Benchmark created:
  Queries: 1000
  Corpus:  1000
[ml] Processed benchmark not found.
[ml] Loading from Hugging Face...
[ml] Benchmark created:
  Queries: 1000
  Corpus:  1000
[mr] Processed benchmark not found.
[mr] Loading from Hugging Face...
[mr] Benchmark created:
  Queries: 1000
  Corpus:  1000
[ne] Processed benchmark not found.
[ne] Loading from Hugging Face...
[ne] Benchmark created:
  Queries: 1000
  Corpus:  1000
[or] Processed benchmark not found.
[or] Loading from Hugging Fa

In [3]:
q, c, gold = bench["hi"]

print("Queries:", len(q))
print("Documents:", len(c))
print("Gold labels:", len(gold))

print("\nExample query:")
print(q["query"].iloc[0])

print("\nExample passage:")
print(c["text"].iloc[0])

Queries: 1000
Documents: 1000
Gold labels: 1000

Example query:
हिरलूम टमाटर का क्या अर्थ है

Example passage:
डलदल की पूर्ण परिभाषा 1 एक आर्द्रभूमि जो अक्सर आंशिक या अस्थायी रूप से पानी से ढकी होती है, विशेष रूप से एक जो लकड़ी की वनस्पतियों से प्रभुत्व होती है 2 डलदल का एक टुकड़ा 3 एक कठिन या परेशान करने वाली स्थिति या विषय डलदल विशेषण


In [4]:
print(tokenize("रक्त प्रकार और जनसंख्या। ओ पॉजिटिव"))

['रक्त', 'प्रकार', 'और', 'जनसंख्या', 'ओ', 'पॉजिटिव']


In [5]:
q, c, gold = bench["hi"]

t0 = time.time()

doc_tokens = [
    tokenize(text)
    for text in c["text"]
]

query_tokens = [
    tokenize(text)
    for text in q["query"]
]

bm25 = BM25(
    k1=1.5,
    b=0.75,
)

bm25.fit(doc_tokens)

S = bm25.score(query_tokens)

print("Score matrix:", S.shape)

metrics = record(
    "bm25",
    "hi",
    S,
    gold,
)

print(metrics)

print(
    f"Time: {(time.time() - t0):.2f} seconds"
)

Score matrix: (1000, 1000)
{'MRR': 0.7116974055671412, 'MRR@10': 0.7087182539682539, 'R@1': 0.637, 'R@5': 0.8, 'R@10': 0.842, 'R@20': 0.866, 'R@100': 0.904, 'nDCG@10': 0.7412287107423573, 'n_queries': 1000, 'model': 'bm25', 'key': 'hi', 'setting': 'mono', 'n_docs': 1000}
Time: 0.51 seconds


In [6]:
gold_shuffled = np.random.default_rng(
    1
).permutation(gold)

real_mrr = metrics_from_ranks(
    gold_ranks(S, gold)
)["MRR"]

shuffled_mrr = metrics_from_ranks(
    gold_ranks(S, gold_shuffled)
)["MRR"]

print("Real BM25 MRR     :", round(real_mrr, 4))
print("Shuffled-gold MRR :", round(shuffled_mrr, 4))

Real BM25 MRR     : 0.7117
Shuffled-gold MRR : 0.0074


In [7]:
for lang in LANGS:

    q, c, gold = bench[lang]

    t0 = time.time()

    doc_tokens = [
        tokenize(text)
        for text in c["text"]
    ]

    query_tokens = [
        tokenize(text)
        for text in q["query"]
    ]

    bm25 = BM25(
        k1=1.5,
        b=0.75,
    )

    bm25.fit(doc_tokens)

    S = bm25.score(query_tokens)

    metrics = record(
        "bm25",
        lang,
        S,
        gold,
    )

    print(
        f"{lang} | "
        f"MRR={metrics['MRR']:.4f} | "
        f"R@1={metrics['R@1']:.4f} | "
        f"R@10={metrics['R@10']:.4f} | "
        f"R@100={metrics['R@100']:.4f} | "
        f"{time.time() - t0:.1f}s"
    )

as | MRR=0.4616 | R@1=0.3800 | R@10=0.5990 | R@100=0.7200 | 0.4s
bn | MRR=0.6189 | R@1=0.5370 | R@10=0.7720 | R@100=0.8510 | 0.4s
gu | MRR=0.5465 | R@1=0.4685 | R@10=0.6877 | R@100=0.7998 | 0.3s
hi | MRR=0.7117 | R@1=0.6370 | R@10=0.8420 | R@100=0.9040 | 0.8s
kn | MRR=0.5380 | R@1=0.4760 | R@10=0.6630 | R@100=0.7170 | 0.7s
ml | MRR=0.5248 | R@1=0.4550 | R@10=0.6620 | R@100=0.7300 | 0.8s
mr | MRR=0.5097 | R@1=0.4320 | R@10=0.6630 | R@100=0.7720 | 0.5s
ne | MRR=0.5052 | R@1=0.4280 | R@10=0.6610 | R@100=0.7700 | 0.3s
or | MRR=0.5172 | R@1=0.4480 | R@10=0.6520 | R@100=0.7660 | 0.4s
pa | MRR=0.6532 | R@1=0.5750 | R@10=0.7990 | R@100=0.8820 | 0.3s
ta | MRR=0.5255 | R@1=0.4560 | R@10=0.6790 | R@100=0.7770 | 0.4s
te | MRR=0.5814 | R@1=0.4900 | R@10=0.7530 | R@100=0.8240 | 0.3s
ur | MRR=0.6529 | R@1=0.5800 | R@10=0.7890 | R@100=0.8810 | 0.3s
